In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
import calendar

"""
PATHS
"""

data_dir  = Path()
out_dir   = Path()
out_dir.mkdir(parents=True, exist_ok=True)

tasmax_nc = data_dir / "tasmax_hadukgrid_uk_country_day_19310101-20241231.nc"
tasmin_nc = data_dir / "tasmin_hadukgrid_uk_country_day_19310101-20241231.nc"
out_file  = out_dir  / "weather_hist_daily.parquet"

start_date = "2010-01-01"
end_date   = "2024-12-31"

In [ ]:
"""
VERSION INFORMATION
"""

%load_ext watermark

%watermark --python --packages numpy,pandas,xarray,jupyterlab --machine --iversions

### TASK A - weather_hist_daily.parquet  

In [ ]:
"""
LOAD DATA AND DECODE GEO_REGIONS
"""

tasmax_ds = xr.open_dataset(tasmax_nc)
tasmin_ds = xr.open_dataset(tasmin_nc)

regions_raw = tasmax_ds["geo_region"].values
regions_str = [r.decode("utf-8").strip() for r in regions_raw]
print(f"Available regions: {regions_str}")

In [ ]:
"""
SELECT REQUIRED REGIONS (ENG & WALES, SCOTLAND)
"""

regions = {
    "England and Wales" : "eng_wales",
    "Scotland" : "scotland"
}

# Find the integer index of each target region
region_indices = {}
for label in regions:
    if label not in regions_str:
        raise ValueError(f"Expected region '{label}' not found in NetCDF. "
                         f"Available: {regions_str}")
    region_indices[label] = regions_str.index(label)

print(f"Region indices selected: {region_indices}")

In [ ]:
"""
RESTRICT TO CORRECT TIME PREIOD
"""

tasmax_ds = tasmax_ds.sel(time=slice(start_date, end_date))
tasmin_ds = tasmin_ds.sel(time=slice(start_date, end_date))

print(f"Time range in file: {str(tasmax_ds.time.values[0])[:10]} "
      f"to {str(tasmax_ds.time.values[-1])[:10]}")

In [ ]:
"""
BUILD DATAFRAME
"""

frames = []

for label, slug in regions.items():
    idx = region_indices[label]

    # Extract 1-D arrays for this region
    tasmax_vals = tasmax_ds["tasmax"].isel(region=idx).values
    tasmin_vals = tasmin_ds["tasmin"].isel(region=idx).values

    # Strip the 12:00:00 noon timestamp to pure date
    dates = pd.to_datetime(tasmax_ds["time"].values).normalize()

    df = pd.DataFrame({
        "date":     dates,
        "region":   slug,
        "tasmin_c": tasmin_vals,
        "tasmax_c": tasmax_vals,
    })
    frames.append(df)

df = pd.concat(frames, ignore_index=True)
df["tmean_c"] = (df["tasmax_c"] + df["tasmin_c"]) / 2
df = df[["date", "region", "tasmin_c", "tasmax_c", "tmean_c"]]
print(f"DataFrame shape: {df.shape}")

In [ ]:
"""
QA CHECKS
"""

# No duplicate (date, region) pairs
n_dups = df.duplicated(subset=["date", "region"]).sum()
if n_dups > 0:
    raise AssertionError(f"QA FAIL: {n_dups} duplicate (date, region) rows found.")

# No missing values
n_null = df[["tasmin_c", "tasmax_c", "tmean_c"]].isna().sum().sum()
if n_null > 0:
    raise AssertionError(f"QA FAIL: {n_null} missing temperature values found.")

# No missing dates in either region
expected_dates = pd.date_range(start_date, end_date, freq="D")
for slug in regions.values():
    actual = set(df[df["region"] == slug]["date"])
    missing = [d for d in expected_dates if d not in actual]
    if missing:
        raise AssertionError(
            f"QA FAIL: {len(missing)} missing dates in region '{slug}'. "
            f"First few: {missing[:5]}"
        )

# Plausible temperature range (-30 to 50 is generous but catches bad data)
for col in ["tasmin_c", "tasmax_c", "tmean_c"]:
    if df[col].min() < -30 or df[col].max() > 50:
        raise AssertionError(
            f"QA FAIL: '{col}' out of plausible range. "
            f"Got {df[col].min():.1f} to {df[col].max():.1f} degrees celsius."
        )

print("----- No duplicate keys")
print("----- No missing dates")
print("----- No null values")
print(f"Temperature ranges OK — "
      f"tasmin: {df['tasmin_c'].min():.1f}–{df['tasmin_c'].max():.1f} degrees celsius, "
      f"tasmax: {df['tasmax_c'].min():.1f}–{df['tasmax_c'].max():.1f} degrees celsius.")

In [ ]:
"""
SAVE OUTPUT
"""

df.to_parquet(out_file, index=False)
print(f"\nSaved: {out_file}  ({len(df):,} rows)")

### TASK B - weather_future_daily_ukcp18.parquet  

In [ ]:
# 360-day calendar notice
# UKCP18 regional projections use a 360-day calendar (every month has exactly
# 30 days). As a result, the following Gregorian calendar dates are absent from
# the future dataset every year:
#   - 31st of Jan, Mar, May, Jul, Aug, Oct, Dec (7 days/year)
#   - Feb 28/29 replaced by Feb 30 (differs from Gregorian)
# This amounts to roughly 200 missing Gregorian dates across 2010–2050.
#
# The date column is stored as a plain string (YYYY-MM-DD) to avoid pandas
# crashing on dates like 2010-02-30.
#
# DOWNSTREAM ACTION REQUIRED: before joining this dataset against any
# Gregorian-calendar data (e.g. HadUK historical, demand timeseries),
# a calendar alignment strategy must be agreed. Options include:
#   a) Interpolate missing days in UKCP18 to fill Gregorian gaps
#   b) Drop the corresponding days from the Gregorian dataset
#   c) Aggregate to monthly resolution before joining (sidesteps the issue)

# Choosing to interpolate

In [ ]:
"""
PATHS AND VARIABLE MAP
"""
import re

# Map each filename explicitly to its (region_slug, var_col)
# Still verify against the internal metadata header as a cross-check
csv_files = {
    data_dir / "Eng_Wales_Max_Temp_2010_2050.csv":  ("eng_wales", "tasmax_c"),
    data_dir / "Eng_Wales_Min_Temp_2010_2050.csv":  ("eng_wales", "tasmin_c"),
    data_dir / "Eng_Wales_Mean_Temp_2010_2050.csv": ("eng_wales", "tmean_c"),
    data_dir / "Scot_Max_Temp_2010_2050.csv":       ("scotland",  "tasmax_c"),
    data_dir / "Scot_Min_Temp_2010_2050.csv":       ("scotland",  "tasmin_c"),
    data_dir / "Scot_Mean_Temp_2010_2050.csv":      ("scotland",  "tmean_c"),
}

# Keep a raw truth archive of UKCP18
raw_out_file = out_dir / "weather_future_daily_ukcp18_raw360.parquet"
# Have a second output - Gregorian daily modelling file
# (within year interpolation by relative position)
aligned_out_file = out_dir / "weather_future_daily_ukcp18.parquet"

# How each Variable string maps to our target column name
variable_map = {
    "Mean":    "tmean_c",
    "Maximum": "tasmax_c",
    "Minimum": "tasmin_c",
}

In [ ]:
"""
METADATA PARSER FUNCTION
"""

def parse_metadata(path, expected_region, expected_var):
    """
    Read the metadata block of a UKCP18 CSV and cross-check it against
    the region and variable expected from the filename
    Returns: (n_skip, source_col_example)
    """
    with open(path) as f:
        lines = [f.readline().strip() for _ in range(13)]

    # Line 1: "header length,13" i.e. skip first 12 rows when loading data
    n_skip = int(lines[0].split(",")[1]) - 1

    # Parse metadata lines into a dict
    meta = {}
    for line in lines[1:n_skip]:
        parts = line.split(",", 1)
        if len(parts) == 2:
            meta[parts[0].strip()] = parts[1].strip()

    # Cross-check region
    area = meta.get("Area", "")
    if area not in regions:
        raise ValueError(f"Unrecognised area '{area}' in metadata of {path.name}.")

    actual_region = regions[area]
    if actual_region != expected_region:
        raise AssertionError(
            f"QA FAIL: filename implies region='{expected_region}' but "
            f"metadata says '{actual_region}' in {path.name}. "
            f"Check the file has not been misnamed."
        )

    # Cross-check variable
    variable_str = meta.get("Variable", "")
    actual_var = None
    for keyword, col_name in variable_map.items():
        if keyword in variable_str:
            actual_var = col_name
            break

    if actual_var is None:
        raise ValueError(
            f"Could not identify variable type from '{variable_str}' "
            f"in {path.name}."
        )
    if actual_var != expected_var:
        raise AssertionError(
            f"QA FAIL: filename implies var='{expected_var}' but "
            f"metadata says '{actual_var}' in {path.name}. "
            f"Check the file has not been misnamed."
        )

    # Grab source column example for lookup table
    with open(path) as f:
        for i, line in enumerate(f):
            if i == n_skip:
                raw_cols = line.strip().split(",")
                source_col_example = raw_cols[1] if len(raw_cols) > 1 else ""
                break

    print(f"{path.name} : metadata checks passed "
          f"(region={actual_region}, var={actual_var})")

    return n_skip, source_col_example

In [ ]:
"""
CSV LOADER FUNCTION
"""

def load_ukcp18_csv(path, expected_region, expected_var):
    """
    Load one UKCP18 CSV into a long-format df with columns:
    date, region, climate_member, value
    Also returns var_col and source_col_example (for the lookup table)
    """
    # Call - passes expected values through for cross-checking
    n_skip, source_col_example = parse_metadata(path, expected_region, expected_var)

    # Load - keep Date as string (360-day calendar, Feb has 30 days)
    df = pd.read_csv(path, skiprows=n_skip, dtype={"Date": str})

    # Rename member columns e.g. "Mean air temperature at 1.5m (°C)(01)" becomes member_01
    rename = {}
    for col in df.columns:
        match = re.search(r"\((\d{2})\)$", col)
        if match:
            rename[col] = f"member_{match.group(1)}"

    df = df.rename(columns=rename).rename(columns={"Date": "date"})

    # Keep only date + member columns
    member_cols = [c for c in df.columns if c.startswith("member_")]
    df = df[["date"] + member_cols]

    # Melt to long format
    df_long = df.melt(
        id_vars="date",
        value_vars=member_cols,
        var_name="climate_member",
        value_name=expected_var,
    )
    df_long["region"] = expected_region

    print(f"{path.name} : region={expected_region}, var={expected_var}, "
          f"members={len(member_cols)}, rows={len(df_long)}")

    return df_long[["date", "region", "climate_member", expected_var]], \
           expected_var, source_col_example

In [ ]:
"""
360-DAY TO GREGORIAN ALIGNMENT
"""

def align_360day_year(group):
    group = group.sort_values("date").reset_index(drop=True)

    year   = int(group["year"].iat[0])
    region = group["region"].iat[0]
    member = group["climate_member"].iat[0]

    if len(group) != 360:
        raise AssertionError(
            f"Expected 360 rows for {region}/{member}/{year}, got {len(group)}."
        )

    target_dates = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")

    x_src = (np.arange(360) + 0.5) / 360.0
    x_tgt = (np.arange(len(target_dates)) + 0.5) / float(len(target_dates))

    out = pd.DataFrame({
        "date": target_dates,
        "region": region,
        "climate_member": member,
    })

    for col in ["tasmin_c", "tasmax_c", "tmean_c"]:
        out[col] = np.interp(x_tgt, x_src, group[col].to_numpy(dtype=float))

    return out

def align_ukcp18_360day_to_gregorian(df_raw):
    work = df_raw.copy()
    work["year"] = work["date"].str[:4].astype(int)

    df_greg = (
        work
        .sort_values(["region", "climate_member", "year", "date"])
        .groupby(["region", "climate_member", "year"], group_keys=False)
        .apply(align_360day_year)
        .reset_index(drop=True)
    )

    return df_greg[["date", "region", "climate_member", "tasmin_c", "tasmax_c", "tmean_c"]]

In [ ]:
"""
LOAD CSVs
"""

var_frames   = {}   # var_col to list of long dfs
source_examples = {}  # member_id to one original column string

for path, (expected_region, expected_var) in csv_files.items():
    df_long, var_col, source_col_example = load_ukcp18_csv(
        path, expected_region, expected_var
    )

    # Collect long frames by variable
    var_frames.setdefault(var_col, []).append(df_long)

    # Record source column example for each member (only needs doing once)
    if not source_examples:
        member_cols = [c for c in df_long.columns if c == "climate_member"]
        for member in df_long["climate_member"].unique():
            num = member.split("_")[1]
            orig = re.sub(r"\(\d{2}\)$", f"({num})", source_col_example)
            source_examples[member] = orig

# Concatenate within each variable (2 regions × 1 file each)
var_combined = {
    var: pd.concat(frames, ignore_index=True)
    for var, frames in var_frames.items()
}

# Merge the three variables on (date, region, climate_member)
print("\nMerging tasmin / tasmax / tmean")

df = var_combined["tasmin_c"]
df = df.merge(var_combined["tasmax_c"], on=["date", "region", "climate_member"],
              how="outer")
df = df.merge(var_combined["tmean_c"],  on=["date", "region", "climate_member"],
              how="outer")

# Sort and enforce column order
df = df.sort_values(["date", "region", "climate_member"]).reset_index(drop=True)
df = df[["date", "region", "climate_member", "tasmin_c", "tasmax_c", "tmean_c"]]

df_raw = df.copy()
df = align_ukcp18_360day_to_gregorian(df_raw)

print(f"----- Raw 360-day shape: {df_raw.shape}")
print(f"----- Gregorian-aligned shape: {df.shape}")
print(f"----- Gregorian date range: {df['date'].min().date()} : {df['date'].max().date()}")

In [ ]:
"""
QA CHECKS
"""

# RAW 360-DAY QA
raw_dups = df_raw.duplicated(subset=["date", "region", "climate_member"]).sum()
if raw_dups > 0:
    raise AssertionError(f"QA FAIL: {raw_dups} duplicate raw (date, region, climate_member) rows.")

raw_nulls = df_raw[["tasmin_c", "tasmax_c", "tmean_c"]].isna().sum()
if raw_nulls.sum() > 0:
    raise AssertionError(f"QA FAIL: missing raw values found:\n{raw_nulls}")

raw_dates_per_year = (
    df_raw.assign(year=df_raw["date"].str[:4].astype(int))
          .groupby(["region", "climate_member", "year"])["date"]
          .nunique()
)
if not (raw_dates_per_year == 360).all():
    raise AssertionError("QA FAIL: raw UKCP18 calendar is not consistently 360-day.")

# GREGORIAN QA
dups = df.duplicated(subset=["date", "region", "climate_member"]).sum()
if dups > 0:
    raise AssertionError(f"QA FAIL: {dups} duplicate Gregorian (date, region, climate_member) rows.")

nulls = df[["tasmin_c", "tasmax_c", "tmean_c"]].isna().sum()
if nulls.sum() > 0:
    raise AssertionError(f"QA FAIL: missing Gregorian values found:\n{nulls}")

days_per_year = (
    df.assign(year=df["date"].dt.year)
      .groupby(["region", "climate_member", "year"])["date"]
      .nunique()
)

expected = pd.Series(
    [366 if calendar.isleap(int(y)) else 365 for _, _, y in days_per_year.index],
    index=days_per_year.index,
)

if not days_per_year.equals(expected):
    bad = pd.DataFrame({"actual": days_per_year, "expected": expected})
    raise AssertionError(f"QA FAIL: Gregorian year lengths incorrect:\n{bad[bad['actual'] != bad['expected']].head()}")

horizon = df[(df["date"] >= "2030-01-01") & (df["date"] <= "2045-12-31")]
horizon = horizon.copy().reset_index(drop=True)
if len(horizon) == 0:
    raise AssertionError("QA FAIL: no rows found in 2030–2045 project horizon.")

print("----- Raw 360-day calendar confirmed")
print("----- Gregorian calendar alignment confirmed")
print("----- No duplicate keys")
print("----- No null values")
print(f"2030–2045 horizon rows: {len(horizon):,}")

In [ ]:
"""
SAVE
"""

df_raw.to_parquet(raw_out_file, index=False)
horizon.to_parquet(aligned_out_file, index=False)

print(f"\nSaved raw UKCP18 file: {raw_out_file} ({len(df_raw):,} rows)")
print(f"Saved Gregorian modelling file: {aligned_out_file} ({len(horizon):,} rows)")

### TASK C - MEMBER LOOKUP  

In [ ]:
# Can just use one file here since all csvs have same 12 members

reference_csv = data_dir / "Eng_Wales_Mean_Temp_2010_2050.csv"

out_file = out_dir / "ukcp18_member_lookup.csv"

In [ ]:
"""
READ HEADER LENGTHS AND COLUMN NAMES
"""

print(f"Reading member columns from: {reference_csv.name}")

with open(reference_csv) as f:
    first_line = f.readline().strip()

n_skip = int(first_line.split(",")[1]) - 1

# Read just the header row to get original column names
with open(reference_csv) as f:
    for i, line in enumerate(f):
        if i == n_skip:
            raw_cols = [c.strip() for c in line.strip().split(",")]
            break

# Drop the first column (Date) - only want member columns
member_raw_cols = raw_cols[1:]
print(f"  Found {len(member_raw_cols)} member columns")

In [ ]:
"""
BUILD LOOKUP TABLE
"""

rows = []
for original_col in member_raw_cols:
    # Extract two digit member number from e.g. "Mean air temperature(01)"
    match = re.search(r"\((\d{2})\)$", original_col)
    if not match:
        raise AssertionError(
            f"QA FAIL: could not extract member number from column "
            f"'{original_col}' in {reference_csv.name}."
        )
    member_num_str = match.group(1) # e.g. "01"
    member_num_int = int(member_num_str) # e.g. 1

    rows.append({
        "climate_member": f"member_{member_num_str}",
        "member_number": member_num_int,
        "source_column_example": original_col,
    })

df_lookup = pd.DataFrame(rows).sort_values("member_number").reset_index(drop=True)
print(f"Built lookup with {len(df_lookup)} members")

In [ ]:
"""
QA CHECKS
"""

# Expected 12 members
if len(df_lookup) != 12:
    raise AssertionError(
        f"QA FAIL: expected 12 members, got {len(df_lookup)}."
    )

# No duplicate member entries
n_dups = df_lookup.duplicated(subset=["climate_member"]).sum()
if n_dups > 0:
    raise AssertionError(
        f"QA FAIL: {n_dups} duplicate climate_member entries in lookup."
    )

# Confirm the known absent members are not present
absent_members = {"member_02", "member_03", "member_14"}
unexpected = absent_members & set(df_lookup["climate_member"])
if unexpected:
    raise AssertionError(
        f"QA FAIL: members {unexpected} should be absent from UKCP18 regional "
        f"ensemble but were found in the lookup table."
    )

print("----- 12 members present")
print("----- No duplicate entries")
print("----- Members 02, 03, 14 correctly absent")
print(df_lookup.to_string(index=False))

In [ ]:
"""
SAVE
"""

df_lookup.to_csv(out_file, index=False)
print(f"\nSaved: {out_file}  ({len(df_lookup)} rows)")

### README CREATION  

In [ ]:
hist_parquet = out_dir / "weather_hist_daily.parquet"
future_raw_parquet = out_dir / "weather_future_daily_ukcp18_raw360.parquet"
future_parquet = out_dir / "weather_future_daily_ukcp18.parquet"
lookup_csv = out_dir / "ukcp18_member_lookup.csv"
readme_out = out_dir / "weather_README.md"

In [ ]:
"""
LOAD OUTPUT FILES TO COMPUTE STATS (CAN BE ADDED TO README IF DESIRED)
"""

df_hist = pd.read_parquet(hist_parquet)
df_future_raw = pd.read_parquet(future_raw_parquet)
df_future = pd.read_parquet(future_parquet)
df_lookup = pd.read_csv(lookup_csv)

# Compute historical stats
hist_rows = len(df_hist)
hist_date_min = df_hist["date"].min().strftime("%Y-%m-%d")
hist_date_max = df_hist["date"].max().strftime("%Y-%m-%d")
hist_regions = sorted(df_hist["region"].unique())
hist_nulls = df_hist[["tasmin_c", "tasmax_c", "tmean_c"]].isna().sum().sum()
hist_dups = df_hist.duplicated(subset=["date", "region"]).sum()
hist_tasmin_rng = (df_hist["tasmin_c"].min(), df_hist["tasmin_c"].max())
hist_tasmax_rng = (df_hist["tasmax_c"].min(), df_hist["tasmax_c"].max())
hist_tmean_rng = (df_hist["tmean_c"].min(),  df_hist["tmean_c"].max())

# Missing dates check per region
expected_dates = pd.date_range(hist_date_min, hist_date_max, freq="D")
hist_missing = {}
for region in hist_regions:
    actual = set(df_hist[df_hist["region"] == region]["date"])
    hist_missing[region] = len([d for d in expected_dates if d not in actual])

# Compute future stats (Gregorian-aligned file)
future_rows = len(df_future)
future_date_min = df_future["date"].min().strftime("%Y-%m-%d")
future_date_max = df_future["date"].max().strftime("%Y-%m-%d")
future_regions = sorted(df_future["region"].unique())
future_members = sorted(df_future["climate_member"].unique())
future_nulls = df_future[["tasmin_c", "tasmax_c", "tmean_c"]].isna().sum().sum()
future_dups = df_future.duplicated(
    subset=["date", "region", "climate_member"]
).sum()

# Raw 360-day calendar check (must use raw file, not aligned file)
dates_per_year = (
    df_future_raw[
        (df_future_raw["region"] == "eng_wales") &
        (df_future_raw["climate_member"] == "member_01")
    ]
    .groupby(df_future_raw.loc[
        (df_future_raw["region"] == "eng_wales") &
        (df_future_raw["climate_member"] == "member_01"),
        "date"
    ].astype(str).str[:4])["date"]
    .nunique()
)
calendar_360_confirmed = (dates_per_year == 360).all()

# Project horizon coverage (Gregorian-aligned file)
horizon_rows = len(
    df_future[
        (df_future["date"] >= "2030-01-01") &
        (df_future["date"] <= "2045-12-31")
    ]
)

print(f"Historical: {hist_rows:,} rows, {hist_date_min} : {hist_date_max}")
print(f"Future (aligned): {future_rows:,} rows, {future_date_min} : {future_date_max}")
print(f"Future raw 360-day confirmed: {calendar_360_confirmed}")
print(f"Lookup: {len(df_lookup)} members")

In [ ]:
readme = f"""# Weather Data Preprocessing — README

**Project:** Small Nuclear Reactor Integration into the GB National Grid  
**Person:** Person 4

---

## 1. Input Files

### HadUK-Grid (historical)
| File | Description |
|---|---|
| `tasmax_hadukgrid_uk_country_day_19310101-20241231.nc` | Daily maximum temperature, 8 UK regions, 1931–2024 |
| `tasmin_hadukgrid_uk_country_day_19310101-20241231.nc` | Daily minimum temperature, same structure |

### UKCP18 (future projections)
| File | Region | Variable |
|---|---|---|
| `Eng_Wales_Max_Temp_2010_2050.csv` | England and Wales | `tasmax_c` |
| `Eng_Wales_Min_Temp_2010_2050.csv` | England and Wales | `tasmin_c` |
| `Eng_Wales_Mean_Temp_2010_2050.csv` | England and Wales | `tmean_c` |
| `Scot_Max_Temp_2010_2050.csv` | Scotland | `tasmax_c` |
| `Scot_Min_Temp_2010_2050.csv` | Scotland | `tasmin_c` |
| `Scot_Mean_Temp_2010_2050.csv` | Scotland | `tmean_c` |

---

## 2. Regions Kept and Excluded

The HadUK-Grid NetCDF contains 8 region aggregates. Only two were retained:

| geo_region label | Action | Output slug |
|---|---|---|
| Channel Islands | Excluded | — |
| England | Excluded | — |
| England and Wales | **Kept** | `eng_wales` |
| Isle of Man | Excluded | — |
| Northern Ireland | Excluded | — |
| Scotland | **Kept** | `scotland` |
| United Kingdom | Excluded | — |
| Wales | Excluded | — |

---

## 3. Output Files and Date Ranges

| Deliverable | Format | Date Range | Purpose |
|---|---|---|---|
| `weather_hist_daily.parquet` | Parquet | 2010-01-01 : 2024-12-31 | Historical daily weather for modelling baseline |
| `weather_future_daily_ukcp18_raw360.parquet` | Parquet | 2010-01-01 : 2049-12-30 | Raw UKCP18 future data preserved on original 360-day calendar |
| `weather_future_daily_ukcp18.parquet` | Parquet | 2030-01-01 : 2045-12-31 | Gregorian daily future weather for downstream modelling and joins |
| `ukcp18_member_lookup.csv` | CSV | — | Lookup table for retained UKCP18 ensemble members |
| `weather_README.md` | Markdown | — | Documentation |

### Intended downstream use
- Use `weather_hist_daily.parquet` for historical model fitting and diagnostics.
- Use `weather_future_daily_ukcp18.parquet` for joins to demand / calendar-based modelling.
- Keep `weather_future_daily_ukcp18_raw360.parquet` as an audit trail of the untouched UKCP18 calendar.

---

## 4. Temperature Column Derivation

### HadUK-Grid
| Column | Source |
|---|---|
| `tasmax_c` | Loaded directly from `tasmax` variable in tasmax NetCDF |
| `tasmin_c` | Loaded directly from `tasmin` variable in tasmin NetCDF |
| `tmean_c` | Computed as `(tasmax_c + tasmin_c) / 2` — no separate mean file available |

The HadUK time axis stores a noon timestamp (`12:00:00`) on every date.  
This was stripped using `.normalize()` to give clean daily dates.

### UKCP18
| Column | Source |
|---|---|
| `tasmax_c` | Loaded from `Eng_Wales_Max_Temp` / `Scot_Max_Temp` CSV |
| `tasmin_c` | Loaded from `Eng_Wales_Min_Temp` / `Scot_Min_Temp` CSV |
| `tmean_c` | Loaded from `Eng_Wales_Mean_Temp` / `Scot_Mean_Temp` CSV — **not** overwritten with `(tasmin_c + tasmax_c) / 2` |

This means the future mean temperature series remains the official UKCP18 mean-temperature variable.

---

## 5. UKCP18 Metadata Header Handling

Each UKCP18 CSV begins with a metadata block. The parsing procedure is:

1. Line 1 contains `header length,N`.
2. `skiprows = N - 1` is passed to `pd.read_csv()` so row `N` becomes the column header.
3. Lines 2 to `N-1` are parsed into a metadata dictionary.
4. The metadata fields `Area` and `Variable` are extracted and cross-checked against the expected values implied by the filename.
5. Any mismatch raises an `AssertionError` immediately.

This prevents the wrong region or variable file from being loaded silently.

---

## 6. 360-Day Calendar Issue and Implemented Fix

### Problem
UKCP18 regional projections use a **360-day calendar**. In this calendar:
- every month has exactly 30 days
- dates such as 31 January, 31 March, and 31 December do not exist
- February has 30 days instead of 28 or 29

That is valid within UKCP18, but it causes problems for this project because the demand model and historical weather data operate on a normal Gregorian daily calendar.

### Why this matters for this project
The project objective is to model future UK energy requirements using seasonal past performance and future climate. That requires the future temperature series to align properly with:
- Gregorian daily demand data
- bank holidays / weekday structure
- historical daily weather series

Leaving the future file on a 360-day calendar would make daily joins unreliable and would distort day-based demand features.

### Fix adopted
The notebook now produces **two** future outputs instead of one:

1. **Raw archive output**  
   `weather_future_daily_ukcp18_raw360.parquet`  
   - preserves the original UKCP18 360-day calendar exactly
   - keeps the source structure for traceability and auditability

2. **Modelling output**  
   `weather_future_daily_ukcp18.parquet`  
   - converts each `(region, climate_member, year)` from the 360-day source calendar to a standard Gregorian daily calendar
   - creates a modelling-ready daily file for downstream joins

### Alignment method used
The alignment is performed by **within-year interpolation by relative position through the year**.

For each `(region, climate_member, year)`:
- the 360 source days are mapped onto fractional positions through the year
- the Gregorian target days (365 or 366 depending on leap year) are mapped onto fractional positions through the same year
- `tasmin_c`, `tasmax_c`, and `tmean_c` are interpolated from the 360-day source positions onto the Gregorian target positions

In effect:
- the raw climate trajectory for each year is retained
- the year is stretched from 360 source days to 365/366 target days
- no Gregorian dates are missing in the aligned modelling file

### Why this fix is appropriate
This was chosen because it is the best fit for the project aims:

- It preserves the original UKCP18 data unchanged in a raw archive.
- It creates a clean daily Gregorian series for direct use in the demand model.
- It avoids dropping valid demand dates from the Gregorian side.
- It avoids forcing the whole project to monthly resolution too early.
- It keeps all ensemble members and both retained regions usable in a consistent way.

### Important interpretation note
The aligned future file is **not** a native Gregorian UKCP18 product. It is a derived daily modelling dataset created from the raw 360-day UKCP18 source via interpolation. For transparency, both files are retained.

---

## 7. UKCP18 Ensemble Members

The retained UKCP18 members are:

`member_01, member_04, member_05, member_06, member_07, member_08, member_09, member_10, member_11, member_12, member_13, member_15`

Members `02`, `03`, and `14` are intentionally absent. This is expected and reflects the UKCP18 ensemble design, not a download or preprocessing error.

---

## 8. QA Summary

### A — `weather_hist_daily.parquet`

| Check | Result |
|---|---|
| Regions retained | `eng_wales`, `scotland` |
| Date column type | Daily datetime, no time component |
| Duplicate (`date`, `region`) keys | 0 |
| Null temperature values | 0 |
| Missing dates within retained historical span | 0 per region |

### B — `weather_future_daily_ukcp18_raw360.parquet`

| Check | Result |
|---|---|
| Regions retained | `eng_wales`, `scotland` |
| Members per region | 12 |
| Calendar type | Original UKCP18 360-day calendar |
| Duplicate (`date`, `region`, `climate_member`) keys | 0 |
| Null temperature values | 0 |
| Dates per `(region, member, year)` | 360 |

### C — `weather_future_daily_ukcp18.parquet`

| Check | Result |
|---|---|
| Regions retained | `eng_wales`, `scotland` |
| Members per region | 12 |
| Calendar type | Gregorian daily calendar |
| Duplicate (`date`, `region`, `climate_member`) keys | 0 |
| Null temperature values | 0 |
| Dates per `(region, member, year)` | 365 or 366 as expected |
| Project horizon 2030–2045 present | Yes |

### D — `ukcp18_member_lookup.csv`

| Check | Result |
|---|---|
| Duplicate entries | 0 |
| Members 02, 03, 14 absent | True |

---

## 9. Practical Notes for the Modelling Team

- Use the **aligned Gregorian** future file for any daily merge with demand, calendar, or historical weather data.
- Use the **raw 360-day** future file only for provenance, validation, or reprocessing.
- Do not recompute `tmean_c` from `tasmin_c` and `tasmax_c` for UKCP18, because a separate UKCP18 mean-temperature file was provided and used directly.
- When checking annual row counts:
  - raw future file should have **360** days per year
  - aligned future file should have **365** or **366** days per year depending on leap year

---

## 10. Acceptance Checklist

- Only `eng_wales` and `scotland` included
- Daily `date` column present in all final outputs
- `tasmin_c`, `tasmax_c`, `tmean_c` present in historical and future outputs
- No duplicate keys in any final output
- Historical file uses Gregorian daily dates
- Raw UKCP18 file preserves the original 360-day calendar
- Aligned UKCP18 file provides a Gregorian daily series for modelling
- 360-day calendar issue is resolved for downstream daily joins
- UKCP18 member exclusions (`02`, `03`, `14`) documented and treated as expected
"""

# ── Save ───────────────────────────────────────────────────────────────────────
readme_out.write_text(readme, encoding="utf-8")
print(f"\nSaved: {readme_out}")